# **PIPELINE NOTES:**

> #### **UPDATES**
> - 09/09/2025: SR created separate analysis to do multi-way interaction volume measure without redundancy and without loss of lower order interation volumes; this can supplement the batch_process_quantification() and batch_summary_stats() functions

**Purpose:** This code addes the segmentations from each organelles channel together to visualize and quantify the total number of pixels in a cell (or image) that overlaps with one, two, three, etc. organelles. This is done without loss of any lower order contacts like what is done in the other analysis.

This analysis should be run using NDCN/infer-subc that is installed using "pip install infer-subc"


# **ImportS**

Run this cell to load the functions necessary for your analysis.

In [ ]:
# regular imports
import numpy as np
import pandas as pd
import time

import tifffile

from pathlib import Path
from typing import Union, List
from infer_subc.utils.batch import list_image_files, find_segmentation_tiff_files
from infer_subc.core.file_io import read_czi_image, read_tiff_image

pd.set_option('display.max_columns', None)


# new function
def quantify_nwayint_vol_nonredundant_noloss(out_file_name: str,
                                        seg_path: Union[Path,str],
                                        out_path: Union[Path, str], 
                                        raw_path: Union[Path,str], 
                                        raw_file_type: str,
                                        organelle_names: List[str],
                                        masks_file_names: List[str],
                                        mask: Union[str, None]=None,
                                        scale:bool=True,
                                        seg_suffix:Union[str, None]=None,
                                        save_interaction_img:bool=False) -> pd.DataFrame :
    """  
    batch process quantification of non-redudant n-way interaction volumes from segmentation tiff files without loss of lower order areas that are part of higher order interaction sites

    Parameters:
    ----------
    out_file_name: str
        the prefix to use when naming the output datatables
    seg_path: Union[Path,str]
        Path or str to the folder that contains the segmentation tiff files
    out_path: Union[Path, str]
        Path or str to the folder that the output datatables will be saved to
    raw_path: Union[Path,str]
        Path or str to the folder that contains the raw image files
    raw_file_type: str
        the file type of the raw data; ex - ".tiff", ".czi"
    organelle_names: List[str]
        a list of all organelle names that will be analyzed; the names should be the same as the suffix used to name each of the tiff segmentation files
        Note: the intensity measurements collect per region (from get_region_morphology_3D function) will only be from channels associated to these organelles 
    region_names: List[str]
        a list of regions, or masks, to measure; the order should correlate to the order of the channels in the "masks" output segmentation file
    masks_file_names: str
        the suffix of the "masks" segmentation file; ex- "masks_B", "masks", etc.
        this function currently does not accept indivial region segmentations 
    mask: Union[str, None]
        the name of the region to use as the mask when measuring the organelles; this should be one of the names listed in regions list; usually this will be the "cell" mask
        if no object is provided, the entire image will be used as the mask
    scale:bool=True
        a tuple that contains the real world dimensions for each dimension in the image (Z, Y, X)
    seg_suffix:Union[str, None]=None
        any additional text that is included in the segmentation tiff files between the file stem and the segmentation suffix
    save_interaction_img:bool=False
        if True, a tiff file will be saved for each image that contains the non-redundant n-way interaction output
        each voxel in the image will be assigned an integer value based on the number of organelles that are present at that voxel
        ex - 1 = 1 organelle present, 2 = 2 organelles present, etc.
        the tiff file will be saved in the out_path location with the name: {out_file_name}_nwayint_img.tiff
    

    Returns:
    ----------
    tab: pd.DataFrame
        a pandas dataframe that contains the quantification results
        this table is also saved as a .csv file in the out_path location
        
    """
    #### TODO: add to batch_quant
    start = time.time()
    img_count = 0

    if isinstance(raw_path, str): raw_path = Path(raw_path)
    if isinstance(seg_path, str): seg_path = Path(seg_path)
    if isinstance(out_path, str): out_path = Path(out_path)
    
    if not Path.exists(out_path):
        Path.mkdir(out_path)
        print(f"making {out_path}")

    # reading list of files from the raw path
    img_file_list = list_image_files(raw_path, raw_file_type)

    # list of segmentation files to collect
    segs_to_collect = organelle_names + masks_file_names

    # containers to collect data tabels
    out_tab = pd.DataFrame()

    for img_f in img_file_list:
        img_count = img_count + 1
        filez = find_segmentation_tiff_files(img_f, segs_to_collect, seg_path, seg_suffix)

        # read in raw file and metadata
        img_data, meta_dict = read_czi_image(filez["raw"])

        # load regions as a list based on order in list (should match order in "masks" file)
        masks = [] 
        for m in masks_file_names:
            mfile = read_tiff_image(filez[m])
            masks.append(mfile)

        # store organelle images as list
        organelles = [read_tiff_image(filez[org]) for org in organelle_names]


        ############################
        ##############################
        ##############################
        #### TODO: add to method_int
        # ADD ALL ORGANELLE SEGS TOGETHER and mask by cell area
        # create empty array to hold all organelle areas & add binary organelle areas together
        all_orgs = np.zeros_like(organelles[0], dtype=np.uint8)
        for o in organelles:
            all_orgs = all_orgs + (o>0)

        if mask is not None:
            mask_img = masks[masks_file_names.index(mask)]
            all_orgs = all_orgs * mask_img.astype(bool)

        ### TODO: put the save step in the single cell function
        if save_interaction_img is True:
            tifffile.imwrite(f"{out_path}/{img_f.stem}_nwayint_img.tiff", all_orgs)
            print(f"all_orgs info: {all_orgs.shape}, {np.unique(all_orgs), {all_orgs.dtype}}")
        
        # measure number of pixels associated to each integer in the all_orgs mask
        vals, count = np.unique(all_orgs, return_counts=True)
        nway_quant = dict(zip(vals, count))
        nway_quant.pop(0)

        # scale the quantification based on voxel size if desired
        if scale is True:
            scale_tup = meta_dict['scale']
        else:
            scale_tup = (1,1,1)

        # apply scaling to the quantification
        for k in nway_quant:
            nway_quant[k] = nway_quant[k] * scale_tup[0] * scale_tup[1] * scale_tup[2]
            
        tab = pd.DataFrame(nway_quant, index=[img_f.stem])
        out_tab = pd.concat([out_tab, tab], axis=0, join='outer')

    # update column names
    if scale is True:
        out_tab = out_tab.add_prefix('volume_with_')
        out_tab = out_tab.add_suffix('_org(s)')
    else:
        out_tab = out_tab.add_prefix('pixel_count_with_')
        out_tab = out_tab.add_suffix('_org(s)')
    
    # insert scale info as first column
    round_scale = (round(scale_tup[0], 4), round(scale_tup[1], 4), round(scale_tup[2], 4))
    out_tab.insert(loc=0, column="scale", value=f"{round_scale}")

    out_tab.to_csv(f"{out_path}/{out_file_name}_nwayint_vol_nonredundant_noloss.csv")

    print(f"Processing of {img_count} images is complete. Total time: {round((time.time()-start)/60,2)} minutes")

    return out_tab

# **Run the analysis**

### **Purpose:**
batch process quantification of non-redudant n-way interaction volumes from segmentation tiff files without loss of lower order areas that are part of higher order interaction sites

### **Input definitions:**
Specify the following information in the code block below. It will be used as inputs in the subsequent function.

    Parameters:
    ----------
    out_file_name: str
        the prefix to use when naming the output datatables
    seg_path: Union[Path,str]
        Path or str to the folder that contains the segmentation tiff files
    out_path: Union[Path, str]
        Path or str to the folder that the output datatables will be saved to
    raw_path: Union[Path,str]
        Path or str to the folder that contains the raw image files
    raw_file_type: str
        the file type of the raw data; ex - ".tiff", ".czi"
    organelle_names: List[str]
        a list of all organelle names that will be analyzed; the names should be the same as the suffix used to name each of the tiff segmentation files
        Note: the intensity measurements collect per region (from get_region_morphology_3D function) will only be from channels associated to these organelles 
    region_names: List[str]
        a list of regions, or masks, to measure; the order should correlate to the order of the channels in the "masks" output segmentation file
    masks_file_names: str
        the suffix of the "masks" segmentation file; ex- "masks_B", "masks", etc.
        this function currently does not accept indivial region segmentations 
    mask: Union[str, None]
        the name of the region to use as the mask when measuring the organelles; this should be one of the names listed in regions list; usually this will be the "cell" mask
        if no object is provided, the entire image will be used as the mask
    scale:bool=True
        a tuple that contains the real world dimensions for each dimension in the image (Z, Y, X)
    seg_suffix:Union[str, None]=None
        any additional text that is included in the segmentation tiff files between the file stem and the segmentation suffix
    save_interaction_img:bool=False
        if True, a tiff file will be saved for each image that contains the non-redundant n-way interaction output
        each voxel in the image will be assigned an integer value based on the number of organelles that are present at that voxel
        ex - 1 = 1 organelle present, 2 = 2 organelles present, etc.
        the tiff file will be saved in the out_path location with the name: {out_file_name}_nwayint_img.tiff
    
### **Output information:**
    Returns:
    ----------
    tab: pd.DataFrame
        a pandas dataframe that contains the quantification results is returned and is printed below
        this table is also saved as a .csv file in the out_path location

In [ ]:
##### USER INPUTS #####
raw_path = Path("C:/Users/Shannon/Documents/Python_Scripts/Infer-subc/raw_two")
raw_file_type = '.czi'
seg_path = Path("C:/Users/Shannon/Documents/Python_Scripts/Infer-subc/out_two")
seg_suffix = '-20230426_test_'
organelle_names = ['LD', 'ER', 'golgi', 'lyso', 'mito', 'perox']
masks_file_names = ['cell', 'nuc']
mask = None
region_names = masks_file_names
scale = False
save_interaction_img = True


### RUN FUNCTION ###
test_tab = quantify_nwayint_vol_nonredundant_noloss(out_file_name = "20250909_testtwo",
                                                    seg_path = seg_path,
                                                    out_path = seg_path, 
                                                    raw_path = raw_path, 
                                                    raw_file_type = raw_file_type,
                                                    organelle_names = organelle_names,
                                                    masks_file_names = masks_file_names,
                                                    mask = mask,
                                                    scale = scale,
                                                    seg_suffix = seg_suffix,
                                                    save_interaction_img=save_interaction_img)
                                                    
test_tab